**OpenAI Luna edition** — upload this file at [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**, then save a copy in Drive.

[Original workshop source](https://github.com/roshanis/notebooks/blob/main/AI_Engineer_World%27s_Fair_Arize_101_Workshop.ipynb) · [Luna API documentation](https://developers.openai.com/api/docs/models/gpt-5.6-luna)

# Getting Started with Arize AX

**Hands-on workshop companion notebook**

This is the runnable notebook for the *Getting Started with Arize AX* workshop. Over the next ~100 minutes we'll build a real AI agent and put it through the full evaluation lifecycle in Arize AX:

1. **Set up tracing** — instrument a Luna agent with Arize AX
2. **Build an agent** — a financial analysis chatbot with web search
3. **Generate test data** — a handful of diverse runs to evaluate
4. **Look at your data** — error analysis *before* writing any evals
5. **Write a code eval** — a deterministic ticker-mention check
6. **Run built-in evals** — correctness vs. faithfulness against the agent's research
7. **Write a custom rubric** — an actionability eval from scratch
8. **Meta-evaluate** — test the judge against human labels you apply in the AX UI
9. **Improve the agent automatically** — feed the judge's explanations back to Luna, then run an experiment to prove the fix worked

Run it in **Google Colab**: choose *File → Save a copy in Drive* so you can edit your own version.

You'll need:
- An **OpenAI API key** ([OpenAI Platform](https://platform.openai.com/))
- An **Arize AX account** (start a free trial at [arize.com](https://arize.com))
  - Your **Arize Space ID** (in your AX workspace settings)
  - An **Arize API key** (generate one in Settings)

> Going to production after the workshop? Online evals, monitors, and the coding-agent improvement loop live in the AX UI and docs ([arize.com/docs/ax](https://arize.com/docs/ax)). The `arize-skills` plugin (`npx skills add Arize-ai/arize-skills`) lets a coding agent do most of this for you.

All model calls default to **gpt-5.6-luna**: research, writing, evaluation, and prompt improvement. The judge uses the same model as the agent; Step 8 measures how well its labels agree with yours.

In Colab's **Secrets** panel, add `openai-api-key`, `arize-api-key`, and `arize-space-id`, and enable notebook access. An OpenAI API key with Luna access and API billing is required. Existing environment variables work too; otherwise setup asks for hidden input. Keys are never saved in this notebook.

Running the notebook sends prompts and results to OpenAI and traces/evaluations to your Arize workspace. The 12-query batch and evaluation steps make paid API calls. Run Step 2 once before the batch. Run Steps 1–7, then apply human labels in Arize before Step 8 and save a failure dataset before the Step 9 experiment. Saved outputs were cleared; results come from your run.

## Step 1: Set Up Tracing

Tracing captures a structured record of every step your agent takes — every LLM call, every tool invocation, every decision — with inputs and outputs at each point. We set it up once, and data flows automatically.

In [ ]:
%pip install -q openai==2.54.0 openinference-instrumentation-openai==0.1.58 arize==8.51.0 arize-otel==0.13.0 arize-phoenix-evals==3.6.0 pandas==2.3.3

In [ ]:
import os
from getpass import getpass
from datetime import datetime, timezone, timedelta

def runtime_secret(env_name, colab_name):
    """Read environment/Colab Secrets, or ask without echoing the value."""
    value = os.environ.get(env_name, "").strip()
    if not value:
        try:
            from google.colab import userdata
        except ImportError:
            userdata = None
        if userdata is not None:
            try:
                value = userdata.get(colab_name)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                value = None
        if not value:
            value = getpass(f"Enter {env_name}: ").strip()
    if not value:
        raise ValueError(f"{env_name} is required.")
    os.environ[env_name] = value
    return value

for env_name, colab_name in (
    ("OPENAI_API_KEY", "openai-api-key"),
    ("ARIZE_API_KEY", "arize-api-key"),
    ("ARIZE_SPACE_ID", "arize-space-id"),
):
    runtime_secret(env_name, colab_name)

AGENT_MODEL = JUDGE_MODEL = IMPROVER_MODEL = "gpt-5.6-luna"
PROJECT_NAME = "aiewf-financial-demo-luna"
RUN_STARTED_AT = datetime.now(timezone.utc) - timedelta(seconds=1)
print(f"Configured {PROJECT_NAME} with {AGENT_MODEL}.")

In [ ]:
from arize.otel import register, Endpoint
from openinference.instrumentation.openai import OpenAIInstrumentor

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name=PROJECT_NAME,
    endpoint=Endpoint.ARIZE,
    batch=False,
)

# Captures both Responses API calls, including web-search response data.
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
from arize import ArizeClient

arize_client = ArizeClient(api_key=os.environ["ARIZE_API_KEY"])
SPACE_ID = os.environ["ARIZE_SPACE_ID"]

## Step 2: Build the Agent

The agent uses OpenAI's Responses API and Luna in two turns:
- **Research** — requires a hosted web search for current financial data.
- **Write** — receives the complete research text and its source URLs, then compiles a report.

Each report gets its own async client, so the notebook and the experiment's worker threads can safely use separate event loops. Both requests use `store=False`; the research is passed explicitly to the writer. The same research is recorded on the report span as `research.context` for faithfulness evaluation.

In [ ]:
from openai import AsyncOpenAI
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

RESEARCH_PROMPT = """Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends.
Include source URLs and dates for the facts you find."""

WRITE_PROMPT = """Now write a concise financial report based on your research above.
Keep source citations and distinguish sourced facts from your analysis."""


def require_response_text(response, stage):
    if response.status != "completed":
        raise RuntimeError(f"The {stage} response did not complete (status={response.status}).")
    text = response.output_text.strip()
    if not text:
        raise RuntimeError(f"The {stage} response contained no text; check for a refusal.")
    return text


def research_context_from_response(response):
    text = require_response_text(response, "research")
    searches = [item for item in response.output if item.type == "web_search_call"]
    if not any(item.status == "completed" for item in searches):
        raise RuntimeError("Research finished without a completed web search.")
    # output_text omits citation metadata. Carry its URLs into the writer's context.
    sources = {}
    for item in response.output:
        if item.type == "message":
            for block in item.content:
                if block.type == "output_text":
                    for annotation in block.annotations:
                        if annotation.type == "url_citation":
                            sources[annotation.url] = annotation.title
    if sources:
        text += "\n\nSources:\n" + "\n".join(
            f"- {title}: {url}" for url, title in sources.items()
        )
    return text


async def _financial_report(tickers, focus, research_prompt, write_prompt, verbose=True):
    if not tickers.strip() or not focus.strip():
        raise ValueError("Both tickers and focus are required.")
    research_input = research_prompt.replace("{tickers}", tickers).replace("{focus}", focus)
    try:
        with tracer.start_as_current_span(
            "financial_report",
            attributes={
                "openinference.span.kind": "CHAIN",
                "input.value": f"Research: {tickers}\nFocus: {focus}",
                "llm.model_name": AGENT_MODEL,
            },
        ) as span:
            async with AsyncOpenAI(timeout=120.0, max_retries=2) as client:
                if verbose:
                    print(f"--- Researching {tickers} ({focus}) ---")
                research_response = await client.responses.create(
                    model=AGENT_MODEL,
                    input=research_input,
                    tools=[{"type": "web_search"}],
                    tool_choice="required",
                    reasoning={"effort": "low"},
                    max_output_tokens=8192,
                    store=False,
                )
                research = research_context_from_response(research_response)
                span.set_attribute("research.context", research)
                if verbose:
                    print(f"  [research] {research[:200]}...")
                    print("--- Writing report ---")
                report_response = await client.responses.create(
                    model=AGENT_MODEL,
                    input=[
                        {"role": "user", "content": research_input},
                        {"role": "assistant", "content": research},
                        {"role": "user", "content": write_prompt},
                    ],
                    reasoning={"effort": "low"},
                    max_output_tokens=8192,
                    store=False,
                )
                report = require_response_text(report_response, "report")
                span.set_attribute("output.value", report)
                return report
    finally:
        # Flush error spans too, so failures remain visible in Arize.
        tracer_provider.force_flush()


async def financial_report(tickers: str, focus: str, verbose: bool = True) -> str:
    return await _financial_report(tickers, focus, RESEARCH_PROMPT, WRITE_PROMPT, verbose)

### Run the agent once to test

This will take a minute or two

In [ ]:
result = await financial_report("TSLA", "financial performance and growth outlook")
print(result)

## Step 3: Generate Test Data

To run meaningful evals, we need more than one trace. Here are 12 test queries covering different tickers, focus areas, and levels of complexity.

In [ ]:
test_queries = [
    {"tickers": "AAPL", "focus": "revenue growth and services segment"},
    {"tickers": "NVDA", "focus": "AI chip demand and valuation metrics"},
    {"tickers": "AMZN", "focus": "AWS performance and profitability"},
    {"tickers": "GOOGL", "focus": "advertising revenue and AI strategy"},
    {"tickers": "MSFT", "focus": "cloud computing segment"},
    {"tickers": "META", "focus": "metaverse investments and ad revenue"},
    {"tickers": "TSLA", "focus": "vehicle deliveries and margins"},
    {"tickers": "RIVN", "focus": "financial health and future growth"},
    {"tickers": "AAPL, MSFT", "focus": "comparative financial analysis"},
    {"tickers": "NVDA", "focus": "competitive landscape and market share"},
    {"tickers": "KO", "focus": "dividend yield and stability"},
    {"tickers": "AMZN", "focus": "profitability trends and outlook"},
]

In [ ]:
for i, query in enumerate(test_queries):
    print(f"[{i+1}/{len(test_queries)}] {query['tickers']} — {query['focus']}")
    await financial_report(query["tickers"], query["focus"], verbose=False)
    print(f"  done")

## Step 4: Look at Your Data — Error Analysis

Before writing a single eval, read your traces. This is the step most tutorials skip — and it's the most important one. You need to understand *what's actually going wrong* before you can measure it.

We'll:
1. **Examine traces** — read the input and output of a few runs
2. **Categorize failures** — label each trace with a root cause
3. **Build a frequency table** — see where to focus first

In [ ]:
import pandas as pd

def prepare_parent_spans(spans):
    """Select completed application reports; key all later joins by span ID."""
    spans = spans.loc[:, ~spans.columns.duplicated()].copy()
    if "context.span_id" not in spans.columns and spans.index.name == "context.span_id":
        spans = spans.reset_index()
    required = {"name", "parent_id", "context.span_id"}
    if not required.issubset(spans.columns):
        raise RuntimeError("No usable report spans yet. Wait for ingestion, then rerun this cell.")
    parents = spans.loc[
        spans["parent_id"].isna() & spans["name"].eq("financial_report")
    ].copy()
    for target, source in (("input", "attributes.input.value"),
                           ("output", "attributes.output.value"),
                           ("context", "attributes.research.context")):
        if source in parents.columns:
            if target in parents.columns:
                parents[target] = parents[target].fillna(parents[source])
            else:
                parents[target] = parents[source]
        elif target not in parents.columns:
            parents[target] = None
    parents = parents.loc[
        parents["input"].fillna("").str.strip().ne("")
        & parents["output"].fillna("").str.strip().ne("")
    ]
    if parents.empty:
        raise RuntimeError("No completed reports found. Run Step 2, wait for ingestion, and retry.")
    if parents["context.span_id"].isna().any():
        raise ValueError("The export contains a report without a span ID.")
    return parents.set_index("context.span_id", verify_integrity=True)


spans_df = arize_client.spans.export_to_df(
    space_id=SPACE_ID,
    project_name=PROJECT_NAME,
    start_time=RUN_STARTED_AT,
    end_time=datetime.now(timezone.utc),
)
parent_spans = prepare_parent_spans(spans_df)
print(f"Found {len(parent_spans)} completed report spans from this session.")

### Examine a few traces

Read the input and a snippet of the output side by side. What jumps out? Look for: hallucinated numbers, missing comparisons, vague recommendations, data that "looks plausible" but you can't verify.

In [ ]:
# Show the first 4 traces: input query and a truncated output
for i, (idx, row) in enumerate(parent_spans.head(4).iterrows()):
    print(f"{'='*60}")
    print(f"TRACE {i+1}")
    print(f"  INPUT:  {row['input']}")
    print(f"  OUTPUT: {str(row['output'])[:300]}...")
    print()

### Categorize failures by root cause

Now label each trace with what you observe. Don't overthink the categories — "looks good," "hallucination," "reasoning gap," "missing data" are fine. The goal is to build intuition about *what kinds of things go wrong*.

In [ ]:
import pandas as pd

# Label each trace with a root cause category.
# Replace these with your own observations after reading the traces above!
trace_categories = {
    "TSLA — financial performance": "looks good",
    "NVDA — AI chip demand": "looks good",
    "AMZN — AWS performance": "possible hallucination",
    "GOOGL — advertising revenue": "looks good",
    "MSFT — cloud computing": "looks good",
    "META — metaverse investments": "reasoning gap",
    "TSLA — vehicle deliveries": "looks good",
    "RIVN — financial health": "unverifiable data",
    "AAPL, MSFT — comparative": "reasoning gap",
    "NVDA — competitive landscape": "possible hallucination",
    "KO — dividend yield": "missing recommendation",
    "AMZN — profitability trends": "possible hallucination",
}

categories_df = pd.DataFrame(
    list(trace_categories.items()),
    columns=["trace", "category"]
)
display(categories_df)

### Frequency table — where should we focus?

Count the categories. The most frequent failure type is where you focus first. But frequency isn't everything — **frequency × severity = priority**. A hallucinated stock price is worse than an awkward sentence.

In [ ]:
# Frequency table of root cause categories
print("Root cause frequency:")
print("-" * 35)
counts = categories_df["category"].value_counts()
for category, count in counts.items():
    bar = "█" * count
    print(f"  {category:<25} {count}  {bar}")
print(f"\nTotal traces: {len(categories_df)}")
print(f"Failure rate: {(len(categories_df) - counts.get('looks good', 0)) / len(categories_df):.0%}")

## Step 5: Code Eval — Ticker Mention Check

Code evals are deterministic functions — no LLM, no API call, no cost. Our simplest useful check: does the output actually mention the ticker we asked about?


1. Click + Add Online Evaluator
2. Give it a name e.g. MentionsEvaluator
3. In `Define imports`, paste this code:

```
from typing import Any, Optional
import re
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    CodeEvaluator,
)
```

4. In `Define Code Evaluator Class` paste this code:

```
class MentionsEvaluator(CodeEvaluator):
    def evaluate(
        self,
        *,
        query: Optional[str] = None,
        report: Optional[str] = None,
        **kwargs: Any,
    ) -> EvaluationResult:
        # Extract ticker symbols from the input (uppercase 1-5 letter words)
        tickers = re.findall(r"\b([A-Z]{1,5})\b", query)
        # Filter to likely tickers (skip common words)
        likely_tickers = [
            t
            for t in tickers
            if len(t) >= 2
            and t not in ("AI", "US", "CEO", "CFO", "IPO", "ETF", "AWS", "USE")
        ]
    
        if not likely_tickers or not report:
            return EvaluationResult(
                label = "unknown",
                score = 0
            )
    
        missing = [t for t in likely_tickers if t not in report.upper()]
    
        if not missing:
            return EvaluationResult(
                label = "pass",
                score = 1
            )
        else:
            return EvaluationResult(
                label = "fail",
                score = 0
            )
```

5. Under `Matching Traces` select `Single trace`
6. Select the input and match it to the `query` variable
7. Select the output and match it to the `report` variable
8. Click Save


### Alternative path: code evals in code

If you don't want to use the UI, here's how you can create and run a code evaluator solely from the notebook. This is the same evaluator as the online evaluator.


In [ ]:
print(f"Using {len(parent_spans)} top-level spans for code evals")

In [ ]:
import re
from phoenix.evals import create_evaluator
from openinference.instrumentation import suppress_tracing
from phoenix.evals import evaluate_dataframe


@create_evaluator(name="mentions_ticker", kind="code")
def mentions_ticker(input, output):
    """Code eval: does the output mention the ticker(s) we asked about?"""
    # Extract ticker symbols from the input (uppercase 1-5 letter words)
    tickers = re.findall(r"\b([A-Z]{1,5})\b", input)
    # Filter to likely tickers (skip common words)
    likely_tickers = [
        t
        for t in tickers
        if len(t) >= 2
        and t not in ("AI", "US", "CEO", "CFO", "IPO", "ETF", "AWS", "USE")
    ]

    if not likely_tickers or not output:
        return {"label": "unknown", "score": 0}

    missing = [t for t in likely_tickers if t not in output.upper()]

    if not missing:
        return {"label": "pass", "score": 1}
    else:
        return {
            "label": "fail",
            "score": 0,
            "explanation": f"Missing tickers: {', '.join(missing)}",
        }


with suppress_tracing():
    results = evaluate_dataframe(dataframe=parent_spans, evaluators=[mentions_ticker])

In [ ]:
# Show all mentions_ticker results (not just head)
import pandas as pd

ticker_scores = pd.json_normalize(results["mentions_ticker_score"])
print("Mentions Ticker Results:")
print(f"  {ticker_scores['label'].value_counts().to_dict()}")
print(f"  {ticker_scores['label'].value_counts().get('pass', 0)}/{len(ticker_scores)} passed")

# Show any failures with explanations
failures = ticker_scores[ticker_scores["label"] == "fail"]
if len(failures) > 0:
    print(f"\nFailures:")
    for _, row in failures.iterrows():
        print(f"  {row.get('explanation', 'no explanation')}")
else:
    print("\nAll passed!")

Send the results back to Arize AX

In [ ]:
def evaluation_scores(eval_results_df, eval_name):
    """Unpack one evaluator's results without losing the original span IDs."""
    if eval_results_df.index.name != "context.span_id" or not eval_results_df.index.is_unique:
        raise ValueError("Evaluation results must have a unique context.span_id index.")
    values = eval_results_df[f"{eval_name}_score"].tolist()
    if not all(isinstance(value, dict) for value in values):
        raise RuntimeError(f"{eval_name} has missing results. Inspect evaluator errors before logging.")
    scores = pd.DataFrame(values, index=eval_results_df.index)
    if "label" not in scores or "score" not in scores or scores[["label", "score"]].isna().any().any():
        raise RuntimeError(f"{eval_name} has incomplete scores; do not log these as passes or failures.")
    if "explanation" not in scores:
        scores["explanation"] = ""
    scores["explanation"] = scores["explanation"].fillna("").astype(str)
    return scores[["label", "score", "explanation"]]


def log_eval_to_ax(eval_results_df, eval_name):
    scores = evaluation_scores(eval_results_df, eval_name)
    annotations = scores.rename(columns={
        column: f"eval.{eval_name}.{column}" for column in scores.columns
    }).reset_index()
    arize_client.spans.update_evaluations(
        space_id=SPACE_ID,
        project_name=PROJECT_NAME,
        dataframe=annotations,
    )
    print(f"Logged {len(annotations)} {eval_name} evaluations to AX")

In [ ]:
log_eval_to_ax(results, eval_name="mentions_ticker")

## Step 6: Built-In Evals

Arize AX ships with pre-built evaluators for common checks. The **Correctness** evaluator uses an LLM judge to assess whether a response is factually accurate, complete, and logically consistent — no prompt engineering required.

> **Note on imports:** the evaluator classes come from `phoenix.evals` (pip package `arize-phoenix-evals`). That is Arize's open-source evaluation library — its import namespace is `phoenix.evals` whether you send data to Arize AX or to open-source Phoenix. The AX docs use these same imports.


In [ ]:
from phoenix.evals.llm import LLM
from phoenix.evals.metrics import CorrectnessEvaluator
from openinference.instrumentation import suppress_tracing

llm = LLM(provider="openai", model=JUDGE_MODEL)
# These are request parameters, passed to evaluators, not the LLM client constructor.
JUDGE_PARAMS = {"reasoning_effort": "low", "max_completion_tokens": 4096}
correctness_eval = CorrectnessEvaluator(llm=llm, **JUDGE_PARAMS)

We suppress tracing here so it doesn't trace our evaluator itself.

In [ ]:
with suppress_tracing():
    correctness_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[correctness_eval]
    )

correctness_results.head()

And send the results to Arize AX again.

In [ ]:
log_eval_to_ax(correctness_results, eval_name="correctness")

### Built-In Eval — Faithfulness

Correctness asks the judge to assess accuracy, but the judge has no live web search in these evaluation calls. Its training knowledge may not cover the agent's recent financial data.

**Faithfulness** checks whether the report is supported by the research provided as **context**. This measures consistency with the research; it does not independently verify that the research itself is accurate. Compare the actual scores from your run.

### Extract research context from traces

The agent records the exact research text and source URLs passed to the writer in its parent span's `research.context` attribute. The export helper maps this to `context`, so we do not depend on child-span ordering or provider-specific message fields.

In [ ]:
has_context = parent_spans["context"].fillna("").str.strip().ne("")
print(f"Research context available for {has_context.sum()}/{len(parent_spans)} reports.")
if not has_context.all():
    raise RuntimeError("Some reports lack research context. Rerun them with the Luna agent before evaluating.")
print(parent_spans["context"].iloc[0][:200] + "...")

### Run the faithfulness evaluator

FaithfulnessEvaluator needs three columns: `input`, `output`, and `context`. We have all three now.

In [ ]:
from phoenix.evals.metrics import FaithfulnessEvaluator

faithfulness_eval = FaithfulnessEvaluator(llm=llm, **JUDGE_PARAMS)
spans_with_context = parent_spans.loc[has_context].copy()
with suppress_tracing():
    faith_results = evaluate_dataframe(
        dataframe=spans_with_context,
        evaluators=[faithfulness_eval],
    )
faith_results.head()

In [ ]:
for name, frame in (("correctness", correctness_results), ("faithfulness", faith_results)):
    scores = evaluation_scores(frame, name)
    print(f"{name}: {scores['label'].value_counts().to_dict()}")
print("Compare disagreements against the research. Faithfulness is not independent fact-checking.")

In [ ]:
log_eval_to_ax(faith_results, eval_name="faithfulness")

## Step 7: Custom Eval Rubric — Actionability

Built-in evals are general-purpose. Custom rubrics check what matters for *your* application. Our financial analyst should produce actionable recommendations — not just summarize data, but tell the reader what to do.

Per the [AX evaluator docs](https://arize.com/docs/ax/evaluate/create-evaluators#tutorial-writing-a-prompt-template), a good prompt template has four parts:
1. **Define the judge's role** — domain context: what system it evaluates, what domain, what its task is
2. **Explicit criteria** — measurable pass/fail conditions, including failure modes from your error analysis
3. **Label the data with XML tags** — wrap each `{variable}` in clear XML tags so the judge knows where instructions end and data begins
4. **Leave output format out of the prompt** — don't tell the judge what labels to emit; the possible responses are defined externally as the evaluator's **Choices** (the `choices=` arg below)

We also add **labeled examples** of each class — not required by the docs, but they measurably improve judge consistency, so we keep them.

In [ ]:
actionability_template = """
You are an expert financial analyst evaluator. Your task is to judge whether
a financial report provides actionable investment guidance, not just raw data.

ACTIONABLE — The report:
- Contains specific recommendations (buy/sell/hold or equivalent guidance)
- Identifies concrete risks with supporting data
- Includes forward-looking analysis, not just historical data
- Provides context for WHY recommendations are made

NOT ACTIONABLE — The report:
- Only summarizes publicly available data without interpretation
- Lacks specific recommendations or next steps
- Presents risks without supporting evidence
- Contains only backward-looking analysis

Here are examples of each:

Example — ACTIONABLE:
\"Based on NVDA's 122% YoY revenue growth driven by data center demand,
strong forward P/E of 35x relative to sector median of 22x, and expanding
margins, NVDA presents a compelling growth position. Key risk: concentration
in AI training chips (~70% of revenue). Recommendation: accumulate on
pullbacks below $800.\"

Example — NOT ACTIONABLE:
\"NVDA is a major player in the semiconductor industry. The company has seen
significant growth in recent years driven by AI demand. NVDA's stock has
performed well. Investors should consider various factors when making
investment decisions.\"

<user_query>
{input}
</user_query>

<financial_report>
{output}
</financial_report>
"""

### Online path

1. In Arize AX, open **Online Evaluators → Add Evaluator**.
2. Select an LLM judge and configure **OpenAI** with your OpenAI API key.
3. Select `gpt-5.6-luna` if your workspace supports it; use the offline path below if it is unavailable in the UI.
4. Give the evaluator a name and set scope to **Trace**.
5. Paste the rubric above and set labels to **actionable (1)** and **not actionable (0)**.
6. Under matching traces, select **single trace** and map `input` and `output`.
7. Save and run the evaluator.

### Offline path

Run our new custom judge.

In [ ]:
from phoenix.evals import ClassificationEvaluator

# ClassificationEvaluator builds a custom LLM-as-judge evaluator
# from a prompt template. The labels live in `choices`, not the prompt.
actionability_evaluator = ClassificationEvaluator(
    name="actionability",
    llm=llm,
    **JUDGE_PARAMS,
    prompt_template=actionability_template,
    choices={"actionable": 1.0, "not actionable": 0.0},
)

with suppress_tracing():
    action_results_df = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[actionability_evaluator]
    )

In [ ]:
import pandas as pd

pd.json_normalize(action_results_df["actionability_score"].head())

Log actionability results back to Arize AX


In [ ]:
log_eval_to_ax(action_results_df, eval_name="actionability")

## Step 8: Meta-Evaluation — Can You Trust Your Judge?

Your LLM judge is a classifier. It makes predictions (actionable / not actionable) that you can check against ground truth — your own human judgment. This is **meta-evaluation**: testing the test.

We'll:
1. **Label a handful of examples in AX** — read the output, apply an annotation in the UI
2. **Pull the labels back** — annotations come back as columns in the span export
3. **Run the judge on the same examples** — see what it says
4. **Compare** — where do they agree? Where do they disagree?
5. **Calculate precision and recall** — put numbers on judge quality

### Label some examples in AX

The easiest place to apply human labels is in the AX UI. AX calls them **annotations**.

**One-time setup** — create an annotation config:
1. Open any trace from the `aiewf-financial-demo-luna` project.
2. Click **Add Annotation** → **+ New Annotation**.
3. Name it `human_actionable`, type **Categorical**, values `actionable` and `not actionable`.

**Then label a handful of examples** (aim for 6+):
- Read the input and the output.
- Apply `actionable` or `not actionable` using the same criteria your rubric uses.
- If you find yourself hesitating, the rubric is ambiguous — note what made you hesitate.

When you've labeled some spans, run the next cell to pull the annotations back.

In [ ]:
# Re-export to obtain human annotations, keeping the original evaluation cohort intact.
annotated_export = arize_client.spans.export_to_df(
    space_id=SPACE_ID,
    project_name=PROJECT_NAME,
    start_time=RUN_STARTED_AT,
    end_time=datetime.now(timezone.utc),
)
annotated_spans = prepare_parent_spans(annotated_export)
ANNOTATION_COL = "annotation.human_actionable.label"
if ANNOTATION_COL not in annotated_spans.columns:
    raise RuntimeError("Apply human_actionable annotations in the Arize UI, then rerun this cell.")
labeled_subset = annotated_spans.loc[annotated_spans[ANNOTATION_COL].notna()].copy()
if labeled_subset.empty:
    raise RuntimeError("No human labels found. Label several reports in Arize and retry.")
normalized = labeled_subset[ANNOTATION_COL].str.replace("_", " ")
if not normalized.isin(["actionable", "not actionable"]).all():
    raise ValueError("Human labels must be actionable or not actionable.")
print(f"Found {len(labeled_subset)} labeled spans.")
display(labeled_subset[["input", ANNOTATION_COL]])

### Run the judge on the same examples

Now run the actionability evaluator on exactly the same traces. Same inputs, same outputs — the only question is whether the judge agrees with you.

In [ ]:
with suppress_tracing():
    judge_results = evaluate_dataframe(
        dataframe=labeled_subset, evaluators=[actionability_evaluator]
    )
judge_labels = evaluation_scores(judge_results, "actionability")
labeled_subset = labeled_subset.drop(
    columns=["judge_label", "judge_explanation"], errors="ignore"
).join(
    judge_labels[["label", "explanation"]].add_prefix("judge_"),
    validate="one_to_one",
)
if labeled_subset["judge_label"].isna().any():
    raise RuntimeError("Some human-labeled spans have no judge result.")

### Compare — where do they agree and disagree?

When the judge disagrees with you, read its explanation. Often it reveals ambiguity in your rubric — that's the most valuable insight from meta-evaluation.

In [ ]:
# Side-by-side comparison: human annotation vs judge prediction
comparison = labeled_subset[["input", ANNOTATION_COL, "judge_label"]].copy()
comparison = comparison.rename(columns={ANNOTATION_COL: "human_label"})
# Normalize the underscore convention — UI annotations use "not_actionable", judge prompt uses "not actionable".
comparison["human_label"] = comparison["human_label"].str.replace("_", " ")
comparison["agree"] = comparison["human_label"] == comparison["judge_label"]

print("Human vs. Judge Comparison")
print("=" * 70)
for i, (idx, row) in enumerate(comparison.iterrows()):
    agree_mark = "✓" if row["agree"] else "✗ DISAGREE"
    print(f"  {row['input'][:45]:<45}  human: {row['human_label']:<15} judge: {row['judge_label']:<15} {agree_mark}")

agreement_rate = comparison["agree"].mean()
print(f"\nAgreement rate: {agreement_rate:.0%} ({comparison['agree'].sum()}/{len(comparison)})")

# Show explanations for disagreements
disagreements = labeled_subset[comparison["human_label"].values != labeled_subset["judge_label"].values]
if len(disagreements) > 0:
    print(f"\n{'='*70}")
    print("DISAGREEMENTS — read the judge's reasoning:")
    for idx, row in disagreements.iterrows():
        human = str(row[ANNOTATION_COL]).replace("_", " ")
        print(f"\n  Query: {row['input'][:60]}")
        print(f"  Human said: {human}")
        print(f"  Judge said: {row['judge_label']}")
        print(f"  Judge's reasoning: {row['judge_explanation'][:300]}...")
else:
    print("\nNo disagreements! The judge matches your labels perfectly.")

### Precision and recall

- **Precision**: When the judge says "not actionable," is it right?
- **Recall**: Of all reports that are truly not actionable, how many does the judge catch?

In most eval scenarios, prioritize **recall** — it's better to flag false positives than to miss real failures.

In [ ]:
# Simple precision/recall calculation for the "not actionable" (fail) class.
# We care most about catching failures, so "not actionable" is our positive class.

human_fail = set(comparison[comparison["human_label"] == "not actionable"].index)
judge_fail = set(comparison[comparison["judge_label"] == "not actionable"].index)

true_positives = len(human_fail & judge_fail)
false_positives = len(judge_fail - human_fail)
false_negatives = len(human_fail - judge_fail)
true_negatives = len(comparison) - true_positives - false_positives - false_negatives

print("Confusion Matrix (for 'not actionable' class)")
print("=" * 45)
print(f"                    Judge: fail  Judge: pass")
print(f"  Human: fail           {true_positives}            {false_negatives}")
print(f"  Human: pass           {false_positives}            {true_negatives}")
print()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

print(f"Precision: {precision:.0%}  (when the judge says 'fail', is it right?)")
print(f"Recall:    {recall:.0%}  (of all real fails, how many does it catch?)")
print(f"\nWith only {len(comparison)} examples, these numbers are rough.")
print(f"For a proper validation, aim for 20-50 labeled examples.")

## Step 9: Improve the Agent — Automatically

The actionability eval told us *what's wrong*, and its explanations told us *why*. Instead of hand-editing the prompts ourselves, we feed those explanations straight back to Luna and let it rewrite the agent's prompts — then run an experiment to prove the new prompts are actually better.

**The loop:**
1. Collect the judge explanations from every failing trace.
2. Hand Luna the current prompts + those explanations + our requirements.
3. Luna returns revised prompts.
4. Run an experiment on the failure dataset — did the scores climb?

Before the experiment cell, save the failure dataset: in the AX UI, filter traces to `actionability == "not actionable"`, click **Save as Dataset**, and name it `aiewf-financial-demo-luna-fails`.

In [ ]:
# Collect the judge explanations from every failing trace, then build a
# single prompt that asks Luna to rewrite the agent's prompts.
# Step 7 already scored every trace — reuse action_results_df.
import pandas as pd

scores = evaluation_scores(action_results_df, "actionability")
scored_reports = parent_spans.join(scores.add_prefix("actionability_"), validate="one_to_one")
if scored_reports["actionability_label"].isna().any():
    raise RuntimeError("Some reports have no actionability score. Rerun Step 7 before improving prompts.")
failing = scored_reports.loc[scored_reports["actionability_label"] == "not actionable"].copy()
print(f"{len(failing)} of {len(parent_spans)} traces failed the actionability eval\n")

REQUIREMENTS = """The financial analysis agent must produce reports that:
- Reference the correct ticker(s) requested by the user
- Include real, recent financial data (ratios, prices, recent news)
- Distinguish forward-looking analysis from historical summary
- Provide actionable recommendations with explicit buy/sell/hold language
- Identify concrete risks with supporting data
- For multi-ticker queries, include a dedicated comparison section"""

explanations = "\n".join(
    f"- {row['actionability_explanation']}" for _, row in failing.iterrows()
)

improvement_prompt = f"""You are improving the prompts of a financial-analysis agent.
The agent runs in two turns: a RESEARCH turn and a WRITE turn.

An LLM judge scored the agent's reports for "actionability" and explained
every failure. Find the RECURRING THEMES across these explanations and
rewrite the two prompts to address them. Keep the changes faithful to the
REQUIREMENTS - improve the product, not just the eval score.

## CURRENT RESEARCH PROMPT
{RESEARCH_PROMPT}

## CURRENT WRITE PROMPT
{WRITE_PROMPT}

## REQUIREMENTS
{REQUIREMENTS}

## JUDGE EXPLANATIONS FOR FAILING REPORTS
{explanations}

Return the two revised prompts, each wrapped in XML tags exactly like this:
<research_prompt>...revised research prompt...</research_prompt>
<write_prompt>...revised write prompt...</write_prompt>
Keep the {{tickers}} and {{focus}} placeholders in the research prompt."""

print(improvement_prompt)

### Ask Luna to rewrite the prompts

The OpenAI Responses API returns the revised prompts. We validate both XML tags and the `{tickers}`/`{focus}` placeholders before using them. This call is suppressed from application tracing. If no reports failed, prompt rewriting and the failure-dataset experiment are skipped.

In [ ]:
import re
from openai import OpenAI

def parse_improved_prompts(reply):
    prompts = []
    for tag in ("research_prompt", "write_prompt"):
        matches = re.findall(fr"<{tag}>(.*?)</{tag}>", reply, re.DOTALL)
        if len(matches) != 1 or not matches[0].strip():
            raise ValueError(f"Expected one nonempty <{tag}> in the prompt-rewrite response.")
        prompts.append(matches[0].strip())
    if not all(token in prompts[0] for token in ("{tickers}", "{focus}")):
        raise ValueError("The revised research prompt must retain {tickers} and {focus}.")
    return tuple(prompts)

if failing.empty:
    IMPROVED_RESEARCH_PROMPT, IMPROVED_WRITE_PROMPT = RESEARCH_PROMPT, WRITE_PROMPT
    print("No failing reports; skipping prompt rewriting and the failure-dataset experiment.")
else:
    with suppress_tracing(), OpenAI(timeout=120.0, max_retries=2) as improvement_client:
        response = improvement_client.responses.create(
            model=IMPROVER_MODEL,
            input=improvement_prompt,
            reasoning={"effort": "low"},
            max_output_tokens=4096,
            store=False,
        )
    reply = require_response_text(response, "prompt rewrite")
    IMPROVED_RESEARCH_PROMPT, IMPROVED_WRITE_PROMPT = parse_improved_prompts(reply)
    print("=== IMPROVED RESEARCH PROMPT ===")
    print(IMPROVED_RESEARCH_PROMPT)
    print("\n=== IMPROVED WRITE PROMPT ===")
    print(IMPROVED_WRITE_PROMPT)

### Wire up the improved agent

Same two-turn agent as Step 2 — it just uses Luna's revised prompts now. We use `.replace()` rather than `.format()` for the placeholders, since an LLM-written prompt may contain other braces.

In [ ]:
async def improved_financial_report(tickers: str, focus: str) -> str:
    return await _financial_report(
        tickers, focus, IMPROVED_RESEARCH_PROMPT, IMPROVED_WRITE_PROMPT, verbose=False
    )

### Define the task and the evaluator

The **task** runs the improved agent on one dataset example. The **evaluator** scores the task's output — we reuse the actionability rubric from Step 7, wrapped so it returns an `EvaluationResult`, the type AX experiments expect.

In [ ]:
import asyncio
import concurrent.futures
from arize.experiments import EvaluationResult


def run_async(coro):
    """Use a fresh worker loop when called from a running notebook event loop."""
    with concurrent.futures.ThreadPoolExecutor(1) as pool:
        return pool.submit(lambda: asyncio.run(coro)).result()


def dataset_input(dataset_row):
    for key in ("attributes.input.value", "input"):
        value = dataset_row.get(key)
        if isinstance(value, str) and value.strip():
            return value
    raise ValueError("Dataset rows need an input or attributes.input.value column.")


def improved_agent_task(dataset_row):
    inp = dataset_input(dataset_row)
    match = re.fullmatch(r"Research: (.+)\nFocus: (.+)", inp, re.DOTALL)
    if not match:
        raise ValueError("Expected dataset input in the form 'Research: TICKER\\nFocus: topic'.")
    return run_async(improved_financial_report(*match.groups()))


def actionability_eval(output, dataset_row):
    with suppress_tracing():
        scores = actionability_evaluator.evaluate({"input": dataset_input(dataset_row), "output": output})
    s = scores[0]
    if s.score is None:
        raise RuntimeError("The experiment judge returned no score.")
    return EvaluationResult(
        score=float(s.score),
        label=s.label,
        explanation=s.explanation or "",
    )

### Run the experiment

`experiments.run` runs the task on every example in the dataset, scores each output with the evaluators, and stores the run in AX. The `dataset` argument takes the dataset name you saved it under in the UI.

In [ ]:
if failing.empty:
    experiment, experiment_df = None, pd.DataFrame()
    print("Skipped: there were no failing reports to put in the failure dataset.")
else:
    experiment, experiment_df = arize_client.experiments.run(
        name="luna-improved-prompts-v1",
        dataset="aiewf-financial-demo-luna-fails",
        space=SPACE_ID,
        task=improved_agent_task,
        evaluators=[actionability_eval],
        concurrency=2,
    )
    print(f"Experiment complete: {len(experiment_df)} examples scored.")
    display(experiment_df)

### Read the results

Each row of `experiment_df` contains a dataset example, the revised agent's output, and its actionability score. Compare examples in Arize's **Experiments** view to see whether the revised prompts helped. Recheck improvements with human labels and a held-out dataset; a higher score from the same model is not proof of better reports.

The loop is: trace → evaluate → inspect explanations → revise prompts → compare experiments.

**Validation status:** package imports, notebook syntax, request construction, tracing, and evaluator adapters were checked offline with mocked responses. Live OpenAI access, hosted web-search results, Arize ingestion, manual annotations, and the saved-dataset experiment must be verified in your account by running the steps above.